# DriverGuard AI - Notebook 02: Eye State CNN Training

This notebook trains the deep convolutional neural network for binary eye state classification (OPEN vs. CLOSED).
Includes data augmentation, learning rate scheduling, early stopping, and model checkpointing.

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from src.models.cnn_model import build_eye_cnn, get_training_callbacks, save_training_metadata

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

### 1. Data Augmentation and Generators

In [ ]:
DATA_DIR = "../data/processed"
IMG_SIZE = (64, 64)
BATCH_SIZE = 64

# Realistic augmentations for driver eye crops
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=10,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.10,
    horizontal_flip=True,
    validation_split=0.20
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

### 2. Build and Compile CNN Architecture

In [ ]:
model = build_eye_cnn(input_shape=(64, 64, 3))
model.summary()

### 3. Model Training with Callbacks

In [ ]:
CHECKPOINT_PATH = "../models/eye_classifier.keras"
callbacks = get_training_callbacks(checkpoint_path=CHECKPOINT_PATH, patience_es=8, patience_lr=3)

EPOCHS = 35
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

# Save training metadata
save_training_metadata(
    output_path="../models/model_metadata.json",
    history_dict=history.history,
    input_shape=(64, 64, 3)
)
print("Training complete! Model saved to:", CHECKPOINT_PATH)